# 02C — Compare + Lock FSP — FINAL READY V4

Membandingkan hasil terbaik 02A dan 02B hanya jika:
- backbone sama;
- fingerprint train+validation sama;
- versi prompt sama;
- keduanya tidak menggunakan test untuk selection;
- keduanya mengaktifkan exclusion terhadap full-source-identical train row.

Ranking final:
**chrF++ → BLEU → mean prompt tokens**.

Output:
- `Config/fsp_locked_config.json`
- `Config/fsp_exact_vs_semantic_comparison.csv`

Jika semantic menang, config final memberi flag `requires_methodology_update=true`.


In [1]:
from pathlib import Path
import os,json
import pandas as pd
from IPython.display import display

PROJECT_ROOT=Path(
    os.getenv("SKRIPSI_PROJECT_ROOT","/workspace/10122233_CikitaMeydilvira_Skripsi")
).expanduser().resolve()
CONFIG_DIR=PROJECT_ROOT/"Config"

exact_path=CONFIG_DIR/"selected_fsp_exact_config.json"
semantic_path=CONFIG_DIR/"selected_fsp_semantic_config.json"

for p in [exact_path,semantic_path]:
    if not p.exists():
        raise FileNotFoundError(f"Belum ada config: {p}")

exact=json.loads(exact_path.read_text(encoding="utf-8"))
semantic=json.loads(semantic_path.read_text(encoding="utf-8"))

for name,obj in [("exact",exact),("semantic",semantic)]:
    required=[
        "method","model_id","dataset_fingerprint","prompt_template_version",
        "train_rows","validation_rows","test_read",
        "validation_chrfpp","validation_bleu","mean_prompt_tokens",
        "identical_full_source_train_rows_excluded_from_retrieval"
    ]
    missing=[x for x in required if x not in obj]
    if missing:
        raise RuntimeError(f"{name}: field hilang {missing}")
    if obj["test_read"] is not False:
        raise RuntimeError(f"{name}: config menyatakan test pernah dibaca.")
    if obj["identical_full_source_train_rows_excluded_from_retrieval"] is not True:
        raise RuntimeError(f"{name}: anti-identical-source retrieval tidak aktif.")

if exact["model_id"]!=semantic["model_id"]:
    raise RuntimeError("Model exact dan semantic berbeda.")
if exact["prompt_template_version"]!=semantic["prompt_template_version"]:
    raise RuntimeError("Versi prompt exact dan semantic berbeda.")
if exact["dataset_fingerprint"]!=semantic["dataset_fingerprint"]:
    raise RuntimeError(
        "Fingerprint dataset exact dan semantic BERBEDA. "
        "Jangan lock; run ulang keduanya pada dataset final yang sama."
    )
if exact["train_rows"]!=semantic["train_rows"] or exact["validation_rows"]!=semantic["validation_rows"]:
    raise RuntimeError("Jumlah train/validation exact-semantic berbeda.")

rows=[]
for obj in [exact,semantic]:
    rows.append({
        "method":obj["method"],
        "chrfpp":float(obj["validation_chrfpp"]),
        "bleu":float(obj["validation_bleu"]),
        "mean_prompt_tokens":float(obj["mean_prompt_tokens"]),
        "paper_status":obj.get("paper_status",""),
    })

df=pd.DataFrame(rows).sort_values(
    ["chrfpp","bleu","mean_prompt_tokens"],
    ascending=[False,False,True],
    kind="stable"
).reset_index(drop=True)
display(df.round(4))

winner_method=df.iloc[0]["method"]
winner=exact if exact["method"]==winner_method else semantic

locked={
    **winner,
    "locked_from_validation":True,
    "final_selection_rule":"chrF++ desc -> BLEU desc -> mean_prompt_tokens asc",
    "compared_methods":[exact["method"],semantic["method"]],
    "test_used_for_selection":False,
    "requires_methodology_update":winner["method"]=="fsp_semantic_fragment_modified",
}

out=CONFIG_DIR/"fsp_locked_config.json"
out.write_text(json.dumps(locked,ensure_ascii=False,indent=2),encoding="utf-8")
df.to_csv(
    CONFIG_DIR/"fsp_exact_vs_semantic_comparison.csv",
    index=False,encoding="utf-8-sig"
)

print("\nFSP LOCKED:")
print(json.dumps(locked,ensure_ascii=False,indent=2))
if locked["requires_methodology_update"]:
    print(
        "\nPERHATIAN: semantic menang. Jika dipakai sebagai FSP final, "
        "Bab 3 harus menyebut varian semantic fragment retrieval."
    )


,method,chrfpp,bleu,mean_prompt_tokens,paper_status
0,fsp_exact_fragment,29.3503,6.6535,1024.7547,paper-like exact fragment matching
1,fsp_semantic_fragment_modified,25.6902,4.6845,725.9057,modified semantic fragment retrieval; not lite...



FSP LOCKED:
{
  "method": "fsp_exact_fragment",
  "paper_status": "paper-like exact fragment matching",
  "model_id": "meta-llama/Llama-3.1-8B-Instruct",
  "dataset_fingerprint": "6c8f3cb81c2dfe100390e26ca33147c8324e90dffa31092ae63c18310ba6fdcc",
  "prompt_template_version": "fsp_duri_indonesia_shared_prompt_v4",
  "source_overlap_count": 1,
  "exact_pair_overlap_count": 0,
  "identical_full_source_train_rows_excluded_from_retrieval": true,
  "train_rows": 417,
  "validation_rows": 53,
  "test_read": false,
  "max_fragment_len": 7,
  "tested_k_values": [
    2,
    4,
    6
  ],
  "k_per_fragment": 6,
  "validation_chrfpp": 29.35026844905274,
  "validation_bleu": 6.653525619586257,
  "mean_prompt_tokens": 1024.754716981132,
  "selection_rule": "complete -> chrF++ desc -> BLEU desc -> mean_prompt_tokens asc",
  "validation_run_dir": "/workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/fsp_exact_validation/20260813T093706_400780Z",
  "locked_from_validation": true,
  "final_selection_rul